## Snapshot por municipio e mês

Criaremos um novo dataframe organizado por municipio e mês do ano. Dessa forma, será feita uma melhor análise de sazonalidade e de padrão de acidentes ao longo dos anos.

No novo dataframe teremos as colunas municipio, ano e mês, assim como as features como quantidade de acidentes, quantidade de feridos, quantidade de mortos, etc.

## Importações e carregamento de dados

In [2]:
import pandas as pd
from pathlib import Path

In [3]:
acidentes_SC = pd.read_parquet(Path('../data/interim/acidentes_SC.parquet'))

## Snapshot

In [4]:
# Extração de ano e mês e agregação inicial por município/mês
acidentes_SC['ano'] = acidentes_SC['data_inversa'].dt.year
acidentes_SC['mes'] = acidentes_SC['data_inversa'].dt.month

# Agrupa as métricas mensais por município
df_agrupado = acidentes_SC.groupby(['municipio', 'ano', 'mes']).agg(
    qtd_acidentes=('id', 'count'),
    qtd_mortos=('mortos', 'sum'),
    qtd_feridos_graves=('feridos_graves', 'sum'),
    qtd_feridos_leves=('feridos_leves', 'sum'),
    total_veiculos_envolvidos=('veiculos', 'sum')
).reset_index()

# Criação do Snapshot Completo (Produto Cartesiano: 12 meses por ano por município)
municipios_unicos = acidentes_SC['municipio'].unique()
anos_unicos = acidentes_SC['ano'].unique()
meses_unicos = list(range(1, 13))

# Gera todas as combinações possíveis
grid_completo = pd.MultiIndex.from_product(
    [municipios_unicos, anos_unicos, meses_unicos],
    names=['municipio', 'ano', 'mes']
).to_frame().reset_index(drop=True)

# Unifica o grid com os dados agregados preenchendo os meses sem acidentes com 0
snapshot_acidentes = pd.merge(
    grid_completo, 
    df_agrupado, 
    on=['municipio', 'ano', 'mes'], 
    how='left'
).fillna(0)

# Reverte tipos numéricos convertidos em float devido ao fillna
colunas_inteiras = [
    'qtd_acidentes', 'qtd_mortos', 'qtd_feridos_graves', 
    'qtd_feridos_leves', 'total_veiculos_envolvidos'
]
snapshot_acidentes[colunas_inteiras] = snapshot_acidentes[colunas_inteiras].astype(int)

# Feature Engineering 

# Converte ano/mês em coluna temporal para ordenação e cálculo de lags
snapshot_acidentes['data_referencia'] = pd.to_datetime(
    snapshot_acidentes['ano'].astype(str) + '-' + snapshot_acidentes['mes'].astype(str) + '-01'
)
snapshot_acidentes = snapshot_acidentes.sort_values(['municipio', 'data_referencia']).reset_index(drop=True)

# Lags Temporais (Histórico de acidentes dos meses anteriores)
snapshot_acidentes['lag_acidentes_1m'] = snapshot_acidentes.groupby('municipio')['qtd_acidentes'].shift(1).fillna(0)
snapshot_acidentes['lag_acidentes_2m'] = snapshot_acidentes.groupby('municipio')['qtd_acidentes'].shift(2).fillna(0)
snapshot_acidentes['lag_acidentes_12m'] = snapshot_acidentes.groupby('municipio')['qtd_acidentes'].shift(12).fillna(0) # Sazonalidade anual

# Médias Móveis (Tendência de curto e médio prazo)
snapshot_acidentes['media_movel_3m'] = snapshot_acidentes.groupby('municipio')['lag_acidentes_1m'].transform(
    lambda x: x.rolling(window=3, min_periods=1).mean()
).fillna(0)

snapshot_acidentes['media_movel_6m'] = snapshot_acidentes.groupby('municipio')['lag_acidentes_1m'].transform(
    lambda x: x.rolling(window=6, min_periods=1).mean()
).fillna(0)

# Atributos Sazonais e Calendário
snapshot_acidentes['eh_alta_temporada'] = snapshot_acidentes['mes'].isin([12, 1, 2, 7]).astype(int) # Verão e férias de julho em SC
snapshot_acidentes['trimestre'] = snapshot_acidentes['data_referencia'].dt.quarter

# Severidade Histórica do Município
snapshot_acidentes['taxa_severidade_acumulada'] = (
    (snapshot_acidentes['qtd_mortos'] + snapshot_acidentes['qtd_feridos_graves']) / 
    (snapshot_acidentes['qtd_acidentes'].replace(0, 1))
)

## Salvamento do snapshot

In [5]:
snapshot_acidentes.to_parquet(Path('../data/processed/snapshot_acidentes.parquet'), index=False)

print(f"Arquivo Parquet salvo com sucesso em data/processed")

Arquivo Parquet salvo com sucesso em data/processed
